# 02 · Entrenamiento de YOLO26l-seg bajo protocolo idéntico con tres semillas

**Proyecto de Investigación II — Protocolo experimental (secciones 7.4, 7.6 y 7.7 del informe)**

Entrena la variante **YOLO26l-seg** (28,0 M parámetros; mAP de máscara 45,5 en COCO val2017, modo e2e) en la tarea de segmentación de instancias de fisuras, con el protocolo común de la Tabla 4.1 del plan y **tres semillas aleatorias (42, 1337, 2024)**, todo lo demás idéntico: la dispersión de métricas entre corridas es atribuible exclusivamente a la semilla (validez interna del diseño, sección 4.5 del informe).

| Hiperparámetro | Valor | Hiperparámetro | Valor |
|---|---|---|---|
| Resolución de entrada | 640 px | Épocas máximas | 100 |
| Tamaño de lote | 16 | Paciencia (early stopping) | 20 |
| Optimizador | SGD | Momentum | 0,937 |
| LR inicial | 0,01 | Caída programada (LRF) | 0,01 |
| Mosaic / MixUp | activo / activo | Inicialización de pesos | COCO |

Requisitos: **Colab con GPU** (T4/L4/A100 recomendado). Cada corrida de 100 épocas sobre 7238 imágenes toma ~40–90 min en T4; el notebook puede ejecutarse por semillas en sesiones separadas (reanudación) y guarda cada corrida en `Drive/grietas/runs/`.

> Nota de variante: si `yolo26l-seg.pt` no está disponible en la versión de Ultralytics instalada, el notebook lo reporta y permite sustituir la variante (parámetro `CANDIDATOS`) — el resultado debe declarar la variante efectivamente entrenada en el registro de corridas.


In [2]:
# ── 1. Entorno, rutas y registro de corridas ─────────────────────────────────
import os, sys, json, subprocess, hashlib, zipfile, shutil
from pathlib import Path
from datetime import datetime

EN_COLAB = 'google.colab' in str(get_ipython())
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/grietas')
else:
    BASE = Path('grietas_local')
    BASE.mkdir(parents=True, exist_ok=True)

DATASET_ZIP = BASE / 'crack_mix_v1.zip'
EXTRACT_DIR = Path('/content/grietas') if EN_COLAB else Path('grietas_extract')
RUNS = BASE / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
REGISTRO = RUNS / 'registro_corridas.json'

def sha256_archivo(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        while True:
            b = f.read(bloque)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def cargar_registro():
    if REGISTRO.exists():
        return json.loads(REGISTRO.read_text(encoding='utf-8'))
    return []

def guardar_registro(reg):
    REGISTRO.write_text(json.dumps(reg, indent=2, ensure_ascii=False), encoding='utf-8')

print('Ruta base :', BASE)
print('Runs      :', RUNS)
print('Registro  :', REGISTRO, '(%d corridas registradas)' % len(cargar_registro()))


Mounted at /content/drive
Ruta base : /content/drive/MyDrive/grietas
Runs      : /content/drive/MyDrive/grietas/runs
Registro  : /content/drive/MyDrive/grietas/runs/registro_corridas.json (3 corridas registradas)


In [2]:
# ── 2. Verificación de GPU y dependencias ──────────────────────────────────────
try:
    import torch
    print('PyTorch %s · CUDA disponible: %s · dispositivo: %s' %
          (torch.__version__, torch.cuda.is_available(),
           torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))
    if not torch.cuda.is_available():
        print('ATENCIÓN: sin GPU el entrenamiento es inviable en tiempo razonable.')
        print('En Colab: Entorno de ejecución -> Cambiar tipo de entorno -> GPU.')
except ImportError:
    print('PyTorch no instalado; Colab lo incluye por defecto.')

try:
    from ultralytics import YOLO
    import ultralytics
    print('Ultralytics instalado, versión:', ultralytics.__version__)
except ImportError:
    print('Instalando Ultralytics...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'],
                   check=True)
    from ultralytics import YOLO
    import ultralytics
    print('Ultralytics instalado, versión:', ultralytics.__version__)


PyTorch 2.14.1+cu130 · CUDA disponible: True · dispositivo: NVIDIA A100-SXM4-80GB
Instalando Ultralytics...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics instalado, versión: 8.4.211


In [3]:
# ── 3. Descompresión del dataset y data.yaml de la clase única fisura ─────────
if DATASET_ZIP.exists():
    HASH_DATASET = sha256_archivo(DATASET_ZIP)
    print('SHA-256 del dataset:', HASH_DATASET)
    if not (EXTRACT_DIR / 'train').exists() and not (EXTRACT_DIR / 'images').exists() and not (EXTRACT_DIR / 'crack_mix_v1' / 'train').exists():
        with zipfile.ZipFile(DATASET_ZIP) as zf:
            zf.extractall(EXTRACT_DIR)
else:
    HASH_DATASET = 'NO_DISPONIBLE'
    print('ADVERTENCIA: no está crack_mix_v1.zip; se asume extracción previa en', EXTRACT_DIR)

def descubrir_estructura(raiz):
    # Si hay un subdirectorio único que contiene la estructura, lo usamos como raíz real
    subdirs = [d for d in raiz.iterdir() if d.is_dir()]
    raices_a_buscar = [raiz]
    if len(subdirs) == 1:
        raices_a_buscar.append(subdirs[0])
    elif any(d.name == 'crack_mix_v1' for d in subdirs):
        raices_a_buscar.append(raiz / 'crack_mix_v1')

    for r_actual in raices_a_buscar:
        encontrados = {}
        for split in ['train', 'val', 'test']:
            for patron in [r_actual / split / 'images', r_actual / 'images' / split,
                           r_actual / 'data' / split / 'images', r_actual / split]:
                if patron.is_dir():
                    lab = None
                    for alt in [patron.parent / 'labels', r_actual / 'labels' / split,
                                r_actual / split / 'labels', r_actual / 'data' / split / 'labels']:
                        if alt.is_dir():
                            lab = alt
                            break
                    encontrados[split] = (str(patron), str(lab) if lab else None)
                    break
        if encontrados:
            return encontrados, r_actual
    return {}, raiz

try:
    SPLITS, RUTA_BASE_EFECTIVA = descubrir_estructura(EXTRACT_DIR)
    assert SPLITS, 'Estructura YOLO no detectada en ' + str(EXTRACT_DIR)
except AssertionError as e:
    print('\n[ERROR]:', e)
    print('\nContenido actual dentro de', EXTRACT_DIR, ':')
    if EXTRACT_DIR.exists():
        for path in sorted(EXTRACT_DIR.rglob('*')):
            if path.is_dir() or path.suffix in ['.yaml', '.json']:
                print('  -', path.relative_to(EXTRACT_DIR))
    else:
        print('  El directorio de extracción no existe.')
    raise e

for s, (i, l) in SPLITS.items():
    print('%-5s -> imágenes %s · etiquetas %s' % (s, i, l))

DATA_YAML = RUNS / 'crack_mix_v1.yaml'
contenido = ['path: ' + str(RUTA_BASE_EFECTIVA)]
for s, (i, _) in SPLITS.items():
    # Guardamos rutas relativas a la ruta base efectiva para máxima portabilidad de YOLO
    rel_i = Path(i).relative_to(RUTA_BASE_EFECTIVA)
    contenido.append('%s: %s' % (s, str(rel_i)))
contenido.append('names:')
contenido.append('  0: fisura')
DATA_YAML.write_text('\n'.join(contenido), encoding='utf-8')
print()
print('data.yaml generado en', DATA_YAML)
print(DATA_YAML.read_text(encoding='utf-8'))

SHA-256 del dataset: e858c3123a0976d9dde71f295375f00fa812e61f71090d670c0528bb392d883b
train -> imágenes /content/grietas/crack_mix_v1/images/train · etiquetas /content/grietas/crack_mix_v1/labels/train
val   -> imágenes /content/grietas/crack_mix_v1/images/val · etiquetas /content/grietas/crack_mix_v1/labels/val
test  -> imágenes /content/grietas/crack_mix_v1/images/test · etiquetas /content/grietas/crack_mix_v1/labels/test

data.yaml generado en /content/drive/MyDrive/grietas/runs/crack_mix_v1.yaml
path: /content/grietas/crack_mix_v1
train: images/train
val: images/val
test: images/test
names:
  0: fisura


In [4]:
# ── 4. Selección de la variante (YOLO26l-seg con respaldo) ─────────────────────
CANDIDATOS = ['yolo26l-seg.pt', 'yolo11l-seg.pt']   # primaria según plan; respaldo documentado

def cargar_modelo():
    errores = []
    for nombre in CANDIDATOS:
        try:
            m = YOLO(nombre)
            print('Variante cargada:', nombre)
            return m, nombre
        except Exception as e:
            errores.append('%s -> %s' % (nombre, str(e)[:90]))
    raise RuntimeError('Ninguna variante disponible:\n' + '\n'.join(errores))

_ , VARIANTE_ACTIVA = cargar_modelo()
print()
print('La corrida se etiquetará con la variante efectiva:', VARIANTE_ACTIVA)


Variante cargada: yolo26l-seg.pt

La corrida se etiquetará con la variante efectiva: yolo26l-seg.pt


In [5]:
# ── 5. Protocolo común (Tabla 4.1 del plan) y semillas ────────────────────────
PROTOCOLO = dict(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.1,
    patience=20,
    cos_lr=False,
    deterministic=True,
    exist_ok=True,
    project=str(RUNS),
    verbose=True,
)
SEEDS = [42, 1337, 2024]
print('Protocolo idéntico por corrida:')
for k, v in PROTOCOLO.items():
    print('  %-14s %s' % (k, v))
print('Semillas:', SEEDS)
print('Nota: los aumentos solo aplican al subconjunto de entrenamiento (sección 4.8.1).')


Protocolo idéntico por corrida:
  data           /content/drive/MyDrive/grietas/runs/crack_mix_v1.yaml
  epochs         100
  imgsz          640
  batch          16
  optimizer      SGD
  lr0            0.01
  momentum       0.937
  lrf            0.01
  mosaic         1.0
  mixup          0.1
  patience       20
  cos_lr         False
  deterministic  True
  exist_ok       True
  project        /content/drive/MyDrive/grietas/runs
  verbose        True
Semillas: [42, 1337, 2024]
Nota: los aumentos solo aplican al subconjunto de entrenamiento (sección 4.8.1).


In [6]:
# ── 6. Entrenamiento: tres corridas bajo protocolo idéntico ───────────────────
# Ejecutar de forma secuencial. Si la sesión de Colab se interrumpe, reejecutar:
# la corrida ya registrada se detecta y se salta (reanudación segura).

def ya_entrenada(semilla):
    return any(r.get('semilla') == semilla and r.get('estado') == 'COMPLETADA'
               for r in cargar_registro())

resultados = {}
for semilla in SEEDS:
    run_id = 'yolo26l_seg_s%d' % semilla
    if ya_entrenada(semilla):
        print('La corrida %s ya está registrada; se omite (elimina su entrada del '
              'registro_corridas.json para reentrenar).' % run_id)
        continue
    print()
    print('=' * 78)
    print('Corrida %s · semilla %d · inicio %s' % (run_id, semilla, datetime.now()))
    print('=' * 78)
    t0 = os.times()
    modelo, variante = cargar_modelo()
    res = modelo.train(seed=semilla, name=run_id, **PROTOCOLO)
    mejores = Path(modelo.trainer.save_dir) / 'weights' / 'best.pt'
    fila = dict(
        run_id=run_id,
        fecha=datetime.now().isoformat(timespec='seconds'),
        semilla=semilla,
        variante=variante,
        hash_datos=HASH_DATASET,
        data_yaml=str(DATA_YAML),
        protocolo={k: v for k, v in PROTOCOLO.items() if k not in ('project', 'verbose')},
        mejor_punto=str(mejores),
        estado='COMPLETADA',
    )
    reg = cargar_registro()
    reg = [r for r in reg if r.get('run_id') != run_id]
    reg.append(fila)
    guardar_registro(reg)
    resultados[semilla] = fila
    print('Corrida %s completada; mejor punto: %s' % (run_id, mejores))

print()
print('Corridas registradas al cierre:', len(cargar_registro()))
for r in cargar_registro():
    print('  %s · semilla %s · %s · %s' % (r['run_id'], r['semilla'], r['variante'], r['estado']))


La corrida yolo26l_seg_s42 ya está registrada; se omite (elimina su entrada del registro_corridas.json para reentrenar).

Corrida yolo26l_seg_s1337 · semilla 1337 · inicio 2026-11-12 19:59:29.959571
Variante cargada: yolo26l-seg.pt
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/grietas/runs/crack_mix_v1.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015

In [3]:
# ── 7. Verificación de los artefactos generados por corrida ─────────────────────
import pandas as pd

filas = []
for r in cargar_registro():
    d = Path(r.get('mejor_punto', '')).parent.parent
    csv = d / 'results.csv'
    epocas = None
    if csv.exists():
        df = pd.read_csv(csv)
        df.columns = [c.strip() for c in df.columns]
        epocas = len(df)
        filas.append(dict(run_id=r['run_id'], semilla=r['semilla'],
                          epocas_efectivas=epocas,
                          ultima_metrica_val=df.iloc[-1].get('metrics/mAP50(B)', None)))
    print('%-18s épocas registradas: %s' % (r['run_id'], epocas))

resumen = pd.DataFrame(filas)
resumen.to_csv(RUNS / 'resumen_entrenamiento.csv', index=False)
print()
print(resumen)
print()
print('Artefactos en Drive:')
for p in sorted(RUNS.glob('yolo26l_seg_s*/weights/best.pt')):
    print('  ', p)


yolo26l_seg_s42    épocas registradas: 100
yolo26l_seg_s1337  épocas registradas: 79
yolo26l_seg_s2024  épocas registradas: 86

              run_id  semilla  epocas_efectivas  ultima_metrica_val
0    yolo26l_seg_s42       42               100             0.88592
1  yolo26l_seg_s1337     1337                79             0.87548
2  yolo26l_seg_s2024     2024                86             0.87412

Artefactos en Drive:
   /content/drive/MyDrive/grietas/runs/yolo26l_seg_s1337/weights/best.pt
   /content/drive/MyDrive/grietas/runs/yolo26l_seg_s2024/weights/best.pt
   /content/drive/MyDrive/grietas/runs/yolo26l_seg_s42/weights/best.pt


## Cierre del notebook 02

Cada corrida deja en `Drive/grietas/runs/yolo26l_seg_s{42,1337,2024}/`: `weights/best.pt`, `results.csv` (curva de aprendizaje por época) y el registro consolidado `registro_corridas.json` (run_id, fecha, semilla, variante, hash SHA-256 del dataset, protocolo y estado) — la evidencia de reproducibilidad de la sección 7.7 del informe.

Siguiente paso: `03_evaluacion_deteccion_segmentacion.ipynb` (evaluación intrínseca sobre las 565 imágenes de prueba congeladas).
